# Assignment 3: CNN classification on CIFAR-10 with PyTorch

This notebook trains convolutional neural networks on CIFAR-10 and contains:
1. the network from the PyTorch CNN tutorial as a baseline,
2. a deeper and wider modified architecture,
3. training on the GPU and a speed comparison with the CPU,
4. logging and visualization with TensorBoard,
5. top-3 test accuracy as the evaluation metric,
6. the top-3 misclassifications displayed in TensorBoard.

The first cell imports the libraries and selects the GPU (`cuda`) if available. The TensorFlow messages in its output ("Could not find cuda drivers") come from TensorBoard's optional TensorFlow import and do not affect PyTorch, which uses the GPU.

In [2]:
import time
import random
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torchvision.transforms import v2
from torch.utils.tensorboard import SummaryWriter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

I0000 00:00:1792701770.670502  161378 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1792701770.956190  161378 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792701772.300306  161378 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


cuda


# 1. Same network as in the PyTorch CNN tutorial
The [PyTorch CIFAR-10 tutorial](https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html) is reproduced unchanged as a baseline.

- **Data:** CIFAR-10, 50,000 training and 10,000 test images (32×32 RGB, 10 classes). Images are converted to tensors and normalized per channel to the range [-1, 1].
- **Model:** two convolutional layers (5×5 kernels, 6 and 16 filters), each followed by ReLU and 2×2 max pooling, then three fully connected layers (120 → 84 → 10). About 62k parameters.
- **Training:** cross-entropy loss, SGD with learning rate 0.001 and momentum 0.9, batch size 4, 2 epochs, on the CPU.
- **Evaluation:** overall top-1 accuracy and accuracy per class on the test set.

**Result:** the baseline reaches **48 %** top-1 accuracy. Ship (77.6 %) and frog (64.2 %) are recognized best; car (22.6 %), deer (26.0 %) and cat (28.3 %) worst.

In [3]:
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 4

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


net = Net()

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)


for epoch in range(2):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 2000 == 1999:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

print('Finished Training')


correct = 0
total = 0
# since we're not training, we don't need to calculate the gradients for our outputs
with torch.no_grad():
    for data in testloader:
        images, labels = data
        # calculate outputs by running images through the network
        outputs = net(images)
        # the class with the highest energy is what we choose as prediction
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %')


# prepare to count predictions for each class
correct_pred = {classname: 0 for classname in classes}
total_pred = {classname: 0 for classname in classes}

# again no gradients needed
with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = net(images)
        _, predictions = torch.max(outputs, 1)
        # collect the correct predictions for each class
        for label, prediction in zip(labels, predictions):
            if label == prediction:
                correct_pred[classes[label]] += 1
            total_pred[classes[label]] += 1


# print accuracy for each class
for classname, correct_count in correct_pred.items():
    accuracy = 100 * float(correct_count) / total_pred[classname]
    print(f'Accuracy for class: {classname:5s} is {accuracy:.1f} %')

[1,  2000] loss: 2.185
[1,  4000] loss: 1.933
[1,  6000] loss: 1.687
[1,  8000] loss: 1.583
[1, 10000] loss: 1.533
[1, 12000] loss: 1.477
[2,  2000] loss: 1.378
[2,  4000] loss: 1.374
[2,  6000] loss: 1.374
[2,  8000] loss: 1.336
[2, 10000] loss: 1.306
[2, 12000] loss: 1.280
Finished Training
Accuracy of the network on the 10000 test images: 48 %
Accuracy for class: plane is 60.5 %
Accuracy for class: car   is 22.6 %
Accuracy for class: bird  is 39.1 %
Accuracy for class: cat   is 28.3 %
Accuracy for class: deer  is 26.0 %
Accuracy for class: dog   is 43.6 %
Accuracy for class: frog  is 64.2 %
Accuracy for class: horse is 62.3 %
Accuracy for class: ship  is 77.6 %
Accuracy for class: truck is 60.3 %


# 2. Architecture change
The network is made deeper and wider:

| Layer | Configuration | Output shape |
|---|---|---|
| conv1 + ReLU | 3 → 8 channels, 3×3, padding 1 | 8 × 32 × 32 |
| conv2 + ReLU + max pool | 8 → 16 channels, 3×3, padding 1 | 16 × 16 × 16 |
| conv3 + ReLU | 16 → 32 channels, 3×3, padding 1 | 32 × 16 × 16 |
| conv4 + ReLU + max pool | 32 → 64 channels, 3×3, padding 1 | 64 × 8 × 8 |
| flatten | | 4096 |
| fc1 + ReLU | 4096 → 4096 | 4096 |
| fc2 + ReLU | 4096 → 1000 | 1000 |
| fc3 | 1000 → 10 | 10 (class scores) |

Small 3×3 kernels with padding keep the spatial resolution inside each block, so four convolutions fit before the image is reduced to 8×8. The model has about **20.9 million parameters** (versus about 62k in the tutorial), almost all of them in fc1 (16.8M) and fc2 (4.1M). This makes training much more expensive, which is what the CPU/GPU comparison in section 3 measures.

In [4]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.conv3 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv4 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 8 * 8, 4096)
        self.fc2 = nn.Linear(4096, 1000)
        self.fc3 = nn.Linear(1000, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(F.relu(self.conv2(x)))
        x = F.relu(self.conv3(x))
        x = self.pool(F.relu(self.conv4(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

## Training and evaluation helpers
- `train(net, device, writer, ...)` moves the model and every batch to `device`, trains with cross-entropy and SGD (momentum 0.9), prints the average loss every `log_every` batches and logs it to TensorBoard as `training loss` (section 4). The x-value is the number of training images seen so far. The function returns the training time in seconds; for the GPU it calls `torch.cuda.synchronize()` first, because CUDA runs asynchronously and the time would otherwise be measured too early.
- `evaluate(net, device, k=3)` reports top-1 and top-k accuracy on the test set (section 5).

In [ ]:
def train(net, device, writer, epochs=2, lr=0.001, loader=None, log_every=2000):
    loader = loader or trainloader
    net.to(device)
    net.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(net.parameters(), lr=lr, momentum=0.9)
    start = time.perf_counter()
    for epoch in range(epochs):
        running_loss = 0.0
        for i, (inputs, labels) in enumerate(loader, 0):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = net(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            if i % log_every == log_every - 1:
                print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / log_every:.3f}')
                writer.add_scalar('training loss', running_loss / log_every,
                                  (epoch * len(loader) + i) * loader.batch_size)
                running_loss = 0.0
    if device.type == 'cuda':
        torch.cuda.synchronize()
    duration = time.perf_counter() - start
    writer.flush()
    print(f'Finished Training on {device} in {duration:.1f} s')
    return duration

## Training on the CPU (baseline for the speedup comparison)
The modified network is trained for 2 epochs on the CPU. `torch.manual_seed(0)` gives the CPU and the GPU model the same initial weights, so the only difference between the two runs is the device.

In [6]:
torch.manual_seed(0)
net_cpu = Net()

# TensorBoard logging (section 4)
cpu_writer = SummaryWriter('runs/cifar10/net_cpu')
cpu_writer.add_graph(net_cpu, images)
cpu_time = train(net_cpu, torch.device('cpu'), cpu_writer)
cpu_writer.close()

/home/debian/jupyter-env/lib/python3.14/site-packages/torch/jit/_trace.py:1000: FutureWarning: `torch.jit.trace` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/home/debian/jupyter-env/lib/python3.14/site-packages/torch/jit/_trace.py:1139: FutureWarning: `torch.jit.trace_method` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


[1,  2000] loss: 2.286
[1,  4000] loss: 1.959
[1,  6000] loss: 1.750
[1,  8000] loss: 1.636
[1, 10000] loss: 1.514
[1, 12000] loss: 1.441
[2,  2000] loss: 1.302
[2,  4000] loss: 1.242
[2,  6000] loss: 1.200
[2,  8000] loss: 1.141
[2, 10000] loss: 1.106
[2, 12000] loss: 1.053
Finished Training on cpu in 1255.7 s


# 3. Train on GPU
The same network, with the same seed and hyperparameters, is trained on the GPU (NVIDIA GeForce RTX 5060 Ti, CUDA). Only the device differs: the model is moved with `net.to(device)` and each batch with `inputs.to(device)`.

| Device | Training time (2 epochs) |
|---|---|
| CPU | 1255.7 s (≈ 21 min) |
| GPU | 105.7 s (≈ 1.8 min) |
| **Speedup** | **11.9×** |

The loss values of both runs agree to about the second decimal place (e.g. 1.053 vs. 1.049 at the end), so the GPU computes the same training, only faster. The small differences come from floating-point operations being executed in a different order on CPU and GPU.

The speedup is limited by the batch size of 4: each step has little work, so the overhead of launching GPU kernels and copying batches dominates. With larger batches the GPU would be considerably faster still.

In [7]:
torch.manual_seed(0)
net = Net()

# TensorBoard logging (section 4)
gpu_writer = SummaryWriter('runs/cifar10/net_gpu')
gpu_writer.add_graph(net, images)
gpu_time = train(net, device, gpu_writer)

print(f'CPU: {cpu_time:.1f} s')
print(f'GPU: {gpu_time:.1f} s')
print(f'Speedup: {cpu_time / gpu_time:.2f}x')

[1,  2000] loss: 2.287
[1,  4000] loss: 1.960
[1,  6000] loss: 1.752
[1,  8000] loss: 1.637
[1, 10000] loss: 1.514
[1, 12000] loss: 1.443
[2,  2000] loss: 1.304
[2,  4000] loss: 1.244
[2,  6000] loss: 1.201
[2,  8000] loss: 1.142
[2, 10000] loss: 1.111
[2, 12000] loss: 1.049
Finished Training on cuda in 105.7 s
CPU: 1255.7 s
GPU: 105.7 s
Speedup: 11.88x


# 5. Top-3 accuracy
The test metric is changed: a prediction counts as correct if the true label is among the three classes with the highest output. `evaluate()` takes the indices of the three largest outputs per image with `outputs.topk(3, dim=1)` and checks whether the label is one of them. Top-1 accuracy is printed as well for comparison.

**Result for the modified network trained on the GPU:**

| Metric | Accuracy |
|---|---|
| Top-1 | 63.98 % |
| Top-3 | **89.62 %** |

The architecture change raises top-1 accuracy from 48 % (tutorial) to 64 % with the same number of epochs. Top-3 accuracy is much higher than top-1 because many errors are confusions between similar classes (e.g. cat/dog, car/truck), where the correct class is usually still the second or third choice.

In [10]:
def evaluate(net, device, k=3):
    net.to(device)
    net.eval()
    top1 = topk = total = 0
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = net(images)
            top = outputs.topk(k, dim=1).indices
            top1 += (top[:, 0] == labels).sum().item()
            topk += (top == labels.unsqueeze(1)).any(dim=1).sum().item()
            total += labels.size(0)
    print(f'Top-1 accuracy: {100 * top1 / total:.2f} %')
    print(f'Top-{k} accuracy: {100 * topk / total:.2f} %')
    return topk / total

evaluate(net, device)

Top-1 accuracy: 63.98 %
Top-3 accuracy: 89.62 %


0.8962

# 6. Top-3 errors in TensorBoard
`top3_errors()` collects every test image whose true label is **not** among the top three predictions and randomly selects five of them. They are written to TensorBoard in three ways:
- `add_figure`: a matplotlib figure with each image, its true label and the three predicted classes (tab *Images*),
- `add_images`: the raw images (tab *Images*),
- `add_text`: the labels and predictions as text (tab *Text*).

The images are un-normalized with `img / 2 + 0.5` so they are shown in their original colors.

In [11]:
def top3_errors(net, device, n=5):
    net.to(device)
    net.eval()
    errors = []
    with torch.no_grad():
        for images, labels in testloader:
            outputs = net(images.to(device)).cpu()
            top = outputs.topk(3, dim=1).indices
            for img, label, pred in zip(images, labels, top):
                if label not in pred:
                    errors.append((img, label.item(), pred.tolist()))
    return random.sample(errors, n)


samples = top3_errors(net, device)

fig = plt.figure(figsize=(15, 4))
for idx, (img, label, pred) in enumerate(samples):
    ax = fig.add_subplot(1, 5, idx + 1, xticks=[], yticks=[])
    ax.imshow(np.transpose((img / 2 + 0.5).numpy(), (1, 2, 0)))
    ax.set_title(f'true: {classes[label]}\npred: {", ".join(classes[p] for p in pred)}')
gpu_writer.add_figure('top-3 errors', fig)
gpu_writer.add_images('top-3 error images', torch.stack([s[0] for s in samples]) / 2 + 0.5)
gpu_writer.add_text('top-3 error labels', '  \n'.join(
    f'{i}: true={classes[l]}, pred={[classes[p] for p in pr]}' for i, (_, l, pr) in enumerate(samples)))
gpu_writer.close()

## Precision-recall curves per class
As an additional analysis, the softmax probabilities on the test set are used to plot one precision-recall curve per class (tab *PR Curves* in TensorBoard). They show how well the network separates each class from all others at every decision threshold, which is more informative than a single accuracy value.

In [ ]:
probs, targets = [], []
net.eval()
with torch.no_grad():
    for images, labels in testloader:
        probs.append(F.softmax(net(images.to(device)), dim=1).cpu())
        targets.append(labels)
probs, targets = torch.cat(probs), torch.cat(targets)

pr_writer = SummaryWriter('runs/cifar10/net_gpu')
for i, classname in enumerate(classes):
    pr_writer.add_pr_curve(classname, targets == i, probs[:, i], global_step=0)
pr_writer.close()

# 4. TensorBoard
All results are logged with `torch.utils.tensorboard.SummaryWriter` into `runs/cifar10/`, one subdirectory per run (`net_cpu`, `net_gpu`):

| TensorBoard tab | Content | Logged in |
|---|---|---|
| Scalars | training loss of the CPU and GPU runs | `train()` via `add_scalar` |
| Graphs | network architecture | `add_graph` before training |
| Images / Text | top-3 errors | section 6 |
| PR Curves | precision-recall curve per class | PR curve cell |

**Reading the loss curves:** with *Step* on the horizontal axis, the CPU and GPU curves lie on top of each other (same seed, same computation). With *Relative* (time) on the horizontal axis, the GPU curve ends after about 0.03 h and the CPU curve only after about 0.3 h, which shows the speedup from section 3. The loss falls from about 2.29 to about 1.05 over the two epochs.

TensorBoard can also be started from a terminal with `tensorboard --logdir runs/cifar10` and opened at http://localhost:6006. The cell below embeds it in the notebook.

In [14]:
%load_ext tensorboard
#%tensorboard --logdir runs/cifar10
from IPython.display import IFrame
IFrame('http://localhost:6006', width='100%', height=800)

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


# Summary

| Model | Device | Training time | Top-1 | Top-3 |
|---|---|---|---|---|
| Tutorial network (~62k parameters) | CPU | n/a | 48 % | n/a |
| Modified network (~20.9M parameters) | CPU | 1255.7 s | n/a | n/a |
| Modified network (~20.9M parameters) | GPU | 105.7 s | 63.98 % | 89.62 % |

- The deeper network improves top-1 accuracy by 16 percentage points with the same training budget (2 epochs).
- Training on the GPU gives identical results about 12× faster.
- Under the top-3 metric almost 90 % of the test images are classified correctly; the remaining errors are mostly visually similar classes.